In [4]:
import pathlib
import os 
import pandas as pd

train = "../Dataset/CXRAY/Landmarks_3_10/train.txt"
val = "../Dataset/CXRAY/Landmarks_3_10/val.txt"

train = pd.read_csv(train, header=None)
val = pd.read_csv(val, header=None)

# merge
train = pd.concat([train, val], ignore_index=True)

train

image_base = "../Dataset/CXRAY/Landmarks_3_10/images"
mask_base = "../Dataset/CXRAY/Landmarks_3_10/masks"

i = 1

image_reference = []

nnUNet_image_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset123_CXRAY/imagesTr"
nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset123_CXRAY/labelsTr"

os.makedirs(nnUNet_image_base, exist_ok=True)
os.makedirs(nnUNet_mask_base, exist_ok=True)

for row in train.iterrows():
    image_path = os.path.join(image_base, row[1][0])
    mask_path = os.path.join(mask_base, row[1][0])

    image_new_name = "image_%s_0000.png" % i
    mask_new_name = "image_%s.png" % i

    image_reference.append("image_%s" % i)

    # copy image and mask
    image_dest = os.path.join(nnUNet_image_base, image_new_name)
    mask_dest = os.path.join(nnUNet_mask_base, mask_new_name)

    os.system("cp %s %s" % (image_path, image_dest))
    os.system("cp %s %s" % (mask_path, mask_dest))

    i += 1

train = train.assign(image_reference=image_reference)
train.to_csv("../cxray_train_with_reference.csv", index=False)

In [8]:
import pathlib
import os 
import pandas as pd

test = "../Dataset/CXRAY/Landmarks_3_10/test.txt"
test = pd.read_csv(test, header=None)

image_base = "../Dataset/CXRAY/Landmarks_3_10/images"
mask_base = "../Dataset/CXRAY/Landmarks_3_10/masks"

i = 1

image_reference = []

nnUNet_image_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset123_CXRAY/imagesTs"
nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset123_CXRAY/labelsTs"
os.makedirs(nnUNet_image_base, exist_ok=True)
os.makedirs(nnUNet_mask_base, exist_ok=True)

for row in test.iterrows():
    image_path = os.path.join(image_base, row[1][0])
    mask_path = os.path.join(mask_base, row[1][0])

    image_new_name = "image_%s_0000.png" % i
    mask_new_name = "image_%s.png" % i

    image_reference.append("image_%s" % i)

    # copy image and mask
    image_dest = os.path.join(nnUNet_image_base, image_new_name)
    mask_dest = os.path.join(nnUNet_mask_base, mask_new_name)

    os.system("cp %s %s" % (image_path, image_dest))
    os.system("cp %s %s" % (mask_path, mask_dest))

    i += 1

test = test.assign(image_reference=image_reference)
test.to_csv("../cxray_test_reference.csv", index=False)

In [9]:
test

,0,image_reference
0,CHNCXR_0157_0.png,image_1
1,CHNCXR_0604_1.png,image_2
2,CHNCXR_0600_1.png,image_3
3,CHNCXR_0613_1.png,image_4
4,CHNCXR_0279_0.png,image_5
...,...,...
176,JPCNN057.png,image_177
177,JPCLN114.png,image_178
178,JPCLN068.png,image_179
179,JPCNN008.png,image_180


In [9]:
from medpy.metric.binary import dc, hd, __surface_distances
import numpy as np
import cv2 
import pandas as pd
import os

def calculate_metrics(pred, gt):
    dc_val = dc(pred, gt)
    hd_val = hd(pred, gt, voxelspacing=(1, 1))
    d1 = __surface_distances(pred, gt, voxelspacing=(1, 1))
    d2 = __surface_distances(gt, pred, voxelspacing=(1, 1))
    assd_val = np.mean(np.concatenate((d1, d2)))
    return dc_val, hd_val, assd_val

test = pd.read_csv("test_reference.csv")

organs = [1,2,3]
organ_names = ['LV Myo', 'LV Endo', 'LA']

nnUNet_pred_base = "CAMUS/nnUNet_raw/predTs"

results = []

test['name'] = test['patient'] + "/" + test['chamber'] + "/" + test['frame']
test['label_path'] = test['image_reference'].apply(lambda x: os.path.join(nnUNet_mask_base, x + ".png"))
test['pred_path'] = test['image_reference'].apply(lambda x: os.path.join(nnUNet_pred_base, x + ".png"))

for row in test.iterrows():
    print(row[1]['name'])

    GT = cv2.imread(row[1]['label_path'], cv2.IMREAD_UNCHANGED)
    mask = cv2.imread(row[1]['pred_path'], cv2.IMREAD_UNCHANGED)

    name = row[1]['name']

    for organ_num, organ_name in zip(organs, organ_names):
        dc_val, hd_val, assd_val = calculate_metrics(mask == int(organ_num), GT == int(organ_num))
        results.append({
            "image": name,
            "organ": organ_name,
            "dc": dc_val,
            "hd": hd_val,
            "assd": assd_val,
        })

results = pd.DataFrame(results)
results.to_csv("Results/CAMUS/nnUNet/results.csv", index=False)

patient0108/2CH/0.png
patient0108/2CH/1.png
patient0108/2CH/2.png
patient0108/2CH/3.png
patient0108/2CH/4.png
patient0108/2CH/5.png
patient0108/2CH/6.png
patient0108/2CH/7.png
patient0108/2CH/8.png
patient0108/2CH/9.png
patient0108/2CH/10.png
patient0108/2CH/11.png
patient0108/2CH/12.png
patient0108/2CH/13.png
patient0108/2CH/14.png
patient0108/2CH/15.png
patient0108/2CH/16.png
patient0108/2CH/17.png
patient0108/2CH/18.png
patient0108/4CH/0.png
patient0108/4CH/1.png
patient0108/4CH/2.png
patient0108/4CH/3.png
patient0108/4CH/4.png
patient0108/4CH/5.png
patient0108/4CH/6.png
patient0108/4CH/7.png
patient0108/4CH/8.png
patient0108/4CH/9.png
patient0108/4CH/10.png
patient0108/4CH/11.png
patient0108/4CH/12.png
patient0108/4CH/13.png
patient0108/4CH/14.png
patient0108/4CH/15.png
patient0108/4CH/16.png
patient0108/4CH/17.png
patient0108/4CH/18.png
patient0108/4CH/19.png
patient0108/4CH/20.png
patient0108/4CH/21.png
patient0108/4CH/22.png
patient0108/4CH/23.png
patient0108/4CH/24.png
patient0